<a href="https://colab.research.google.com/github/Jirakit2533/GitHub/blob/main/Number_Extracter_TrORCAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# CELL 1: ติดตั้งคลังโปรแกรมของ Hugging Face
# ==========================================
!pip install transformers torch pillow -q

import torch
from PIL import Image
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from google.colab import files

print("🔄 กำลังดาวน์โหลดโมเดล TrOCR-Printed จาก Microsoft...")
print("(ไฟล์มีขนาดประมาณ 1.3 GB รอบแรกจะใช้เวลาดาวน์โหลดประมาณ 1-2 นาทีครับ)")

# โหลด Processor (ตัวแปลงภาพ) และ Modelสำหรับตัวอักษรพิมพ์/ดิจิทัล (Printed)
processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-printed')
model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-printed')

print("✅ โหลดโมเดล TrOCR สำเร็จ! ระบบพร้อมทำงานแล้วครับ")

In [ ]:
# ==========================================
# CELL 2: อัปโหลดภาพและดึงข้อความออกเป็น String
# ==========================================
print("📥 เลือกรูปภาพหน้าปัดดิจิทัลของคุณ (แนะนำภาพที่ครอปเฉพาะตัวเลข):")
uploaded = files.upload()

if uploaded:
    file_name = list(uploaded.keys())[0]
    print(f"\n📢 TrOCR กำลังประมวลผลไฟล์: {file_name}")

    # 1. เปิดรูปภาพด้วย Pillow และแปลงเป็นระบบสี RGB
    image = Image.open(file_name).convert("RGB")

    # 2. แปลงรูปภาพให้อยู่ในฟอร์แมตที่ Transformer เข้าใจ (Pixel Values)
    pixel_values = processor(images=image, return_tensors="pt").pixel_values

    # 3. สั่งให้โมเดลเจเนอเรต (ถอดรหัส) ภาพออกมาเป็นอักขระ
    print("🤖 Transformer กำลังแกะรหัสภาพ...")
    with torch.no_grad():
        generated_ids = model.generate(pixel_values)

    # 4. แปลงรหัสอักขระกลับมาเป็น String ภาษาคน
    final_output_string = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

    # ==========================================
    # แสดงผลลัพธ์สุดท้าย
    # ==========================================
    print("\n" + "="*40)
    print(" 🎯 OUTPUT จาก TrOCR (String)")
    print("="*40)
    print(f"👉 '{final_output_string}'")
    print("="*40)
    print(f"ชนิดของข้อมูล: {type(final_output_string)}")

else:
    print("❌ ไม่พบไฟล์ที่อัปโหลด")